# Ablation: Average pooling only

## Change
Use average-pooled vectors only; omit max, min, and signed GeM.

## Kaggle scores
- **Public:** 0.72056 (delta vs baseline: -0.03640)
- **Private:** 0.60662 (delta vs baseline: -0.04171)

## Shared input
Attach dinov3_features.npz generated by features/01_extract_dinov3_7b.ipynb. Only the settings above differ from baseline.

In [ ]:
# CSIRO Image2Biomass Prediction
# Train Ridge multiview stacking models from the frozen DINOv3 feature archive.
# DINOv3 inference is kept in the separate feature-extraction notebook.

import json
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold
from sklearn.multioutput import MultiOutputRegressor
from sklearn.preprocessing import RobustScaler
warnings.filterwarnings("ignore")

class CFG:
    FEATURE_FILE = None
    IMG_SIZES = [256, 512, 768]
    MID_BLOCKS = [35, 38]
    ENSEMBLE_FOLDS = [2, 3, 5]
    POOLING = "avg"
    USE_LOG_SPACE = True
    USE_RAW_SPACE = True
    USE_POST_PROCESS = True
    USE_PCA = True
    PCA_N_COMPONENTS = 355
    PCA_WHITEN = False
    PCA_SVD_SOLVER = "randomized"
    FIT_ON_TRAIN_PLUS_TEST = True
    EXCLUDED_IMAGE_ID = "ID230058600"
    seed = 42
    target_cols = ["Dry_Green_g", "Dry_Dead_g", "Dry_Clover_g", "GDM_g", "Dry_Total_g"]
    target_weights = [0.1, 0.1, 0.1, 0.2, 0.5]

def weighted_r2_metric(y_true, y_pred):
    """Competition metric: weighted R2."""
    weights = np.asarray(CFG.target_weights, dtype=float)
    if y_true.ndim == 1: y_true = y_true.reshape(-1, 1)
    if y_pred.ndim == 1: y_pred = y_pred.reshape(-1, 1)
    ss_res = np.sum(weights * np.sum((y_true - y_pred) ** 2, axis=0))
    global_mean = np.average(np.mean(y_true, axis=0), weights=weights)
    ss_tot = np.sum(weights * np.sum((y_true - global_mean) ** 2, axis=0))
    return 1 - ss_res / (ss_tot + 1e-8)

def post_process_biomass(frame):
    """Project predictions onto biomass target identities and enforce nonnegative values."""
    cols = ["Dry_Green_g", "Dry_Clover_g", "Dry_Dead_g", "GDM_g", "Dry_Total_g"]
    if not all(col in frame.columns for col in cols): return frame
    values = frame[cols].to_numpy().T
    constraints = np.array([[1, 1, 0, -1, 0], [0, 0, 1, 1, -1]], dtype=float)
    try: inverse = np.linalg.inv(constraints @ constraints.T)
    except np.linalg.LinAlgError: inverse = np.linalg.pinv(constraints @ constraints.T)
    projection = np.eye(5) - constraints.T @ inverse @ constraints
    output = frame.copy()
    output[cols] = (projection @ values).T.clip(min=0)
    output["GDM_g"] = output["Dry_Green_g"] + output["Dry_Clover_g"]
    output["Dry_Total_g"] = output["GDM_g"] + output["Dry_Dead_g"]
    return output

def locate_feature_archive():
    if CFG.FEATURE_FILE:
        path = Path(CFG.FEATURE_FILE)
        if not path.is_file(): raise FileNotFoundError(f"Configured feature archive does not exist: {path}")
        return path
    candidates = []
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists(): candidates.extend(kaggle_input.glob("*/dinov3_features.npz"))
    for path in (Path("/kaggle/working/dinov3_features.npz"), Path.cwd() / "dinov3_features.npz"):
        if path.is_file(): candidates.append(path)
    unique = list(dict.fromkeys(path.resolve() for path in candidates))
    if len(unique) != 1:
        found = "\n".join(str(path) for path in unique) or "(none found)"
        raise FileNotFoundError("Expected one dinov3_features.npz. Attach the extraction Notebook output or set CFG.FEATURE_FILE. Found:\n" + found)
    return unique[0]

def select_pool(features, pooling):
    if features.ndim != 2 or features.shape[1] % 4:
        raise ValueError(f"Unexpected pooled feature shape: {features.shape}")
    if pooling == "all": return features
    pool_index = {"avg": 0, "max": 1, "min": 2, "gem_p3_signed": 3}[pooling]
    return features.reshape(features.shape[0], 4, -1)[:, pool_index, :]

def load_feature_archive():
    path = locate_feature_archive()
    print(f"Loading feature archive: {path}")
    with np.load(path, allow_pickle=False) as archive:
        manifest = json.loads(str(archive["manifest_json"].item()))
        train_ids = archive["train_image_ids"].astype(str)
        test_ids = archive["test_image_ids"].astype(str)
        y = archive["y_train"].astype(np.float32)
        if manifest.get("pool_order") != ["avg", "max", "min", "gem_p3_signed"]:
            raise ValueError("Unsupported feature pooling order.")
        if manifest.get("targets") != CFG.target_cols:
            raise ValueError("Archive target columns do not match this notebook.")
        keep = train_ids != CFG.EXCLUDED_IMAGE_ID
        if not keep.any(): raise ValueError("No training rows remain after the configured exclusion.")
        if keep.sum() != len(train_ids): print(f"Excluded {np.sum(~keep)} row(s): {CFG.EXCLUDED_IMAGE_ID}")
        train_ids, y = train_ids[keep], y[keep]
        train_views, test_views = {}, {}
        for size in CFG.IMG_SIZES:
            for layer in ["last"] + [f"b{block}" for block in CFG.MID_BLOCKS]:
                view = f"s{size}_{layer}"
                key_train, key_test = f"train__{view}", f"test__{view}"
                if key_train not in archive or key_test not in archive:
                    raise KeyError(f"Required feature view is missing: {view}")
                train_views[view] = select_pool(archive[key_train][keep].astype(np.float32, copy=False), CFG.POOLING)
                test_views[view] = select_pool(archive[key_test].astype(np.float32, copy=False), CFG.POOLING)
    if len(train_ids) != len(y) or len(test_ids) == 0: raise ValueError("Archive IDs and feature rows are empty or misaligned.")
    return train_ids, test_ids, y, train_views, test_views, manifest

class MultiViewStackingTrainer:
    """Fit one Ridge base model per view and one Ridge meta-model per target."""
    def __init__(self, n_folds=5, seed=42):
        self.n_folds, self.seed = n_folds, seed
    def train_predict(self, X_views, y, X_test_views):
        names = sorted(X_views)
        first = names[0]
        splitter = KFold(n_splits=self.n_folds, shuffle=True, random_state=self.seed)
        oof = {f"{name}_ridge": np.zeros_like(y) for name in names}
        test = {f"{name}_ridge": np.zeros((len(X_test_views[first]), y.shape[1])) for name in names}
        ridge = Ridge(alpha=5.0, random_state=self.seed)
        for fold, (tr, va) in enumerate(splitter.split(X_views[first], y), 1):
            print(f"  Fold {fold}/{self.n_folds}")
            for name in names:
                model = MultiOutputRegressor(clone(ridge)).fit(X_views[name][tr], y[tr])
                oof[f"{name}_ridge"][va] = np.clip(model.predict(X_views[name][va]), 0, None)
                test[f"{name}_ridge"] += np.clip(model.predict(X_test_views[name]), 0, None) / self.n_folds
        return oof, test
    def fit_stacking(self, oof, y, test):
        names = sorted(oof)
        final_oof = np.zeros_like(y)
        final_test = np.zeros((next(iter(test.values())).shape[0], y.shape[1]))
        for target in range(y.shape[1]):
            x_meta = np.column_stack([oof[name][:, target] for name in names])
            x_test = np.column_stack([test[name][:, target] for name in names])
            model = Ridge(alpha=1.0, random_state=self.seed).fit(x_meta, y[:, target])
            final_oof[:, target] = model.predict(x_meta)
            final_test[:, target] = model.predict(x_test)
        return np.clip(final_oof, 0, None), np.clip(final_test, 0, None)

def preprocess_views(train_views, test_views):
    out_train, out_test = {}, {}
    for name in sorted(train_views):
        x_train, x_test = train_views[name], test_views[name]
        scaler = RobustScaler()
        if CFG.FIT_ON_TRAIN_PLUS_TEST:
            scaler.fit(np.vstack([x_train, x_test]))
            x_train, x_test = scaler.transform(x_train), scaler.transform(x_test)
        else:
            x_train, x_test = scaler.fit_transform(x_train), scaler.transform(x_test)
        x_train, x_test = x_train.astype(np.float32, copy=False), x_test.astype(np.float32, copy=False)
        if CFG.USE_PCA:
            n = min(CFG.PCA_N_COMPONENTS, x_train.shape[0] - 1, x_train.shape[1])
            pca = PCA(n_components=n, whiten=CFG.PCA_WHITEN, svd_solver=CFG.PCA_SVD_SOLVER, random_state=CFG.seed)
            x_train, x_test = pca.fit_transform(x_train), pca.transform(x_test)
        out_train[name], out_test[name] = x_train, x_test
    return out_train, out_test

def run_pipeline(folds, train_views, y, test_views, log_space):
    targets = np.log1p(y) if log_space else y
    trainer = MultiViewStackingTrainer(folds, CFG.seed)
    oof, test = trainer.train_predict(train_views, targets, test_views)
    oof, test = trainer.fit_stacking(oof, targets, test)
    if log_space: oof, test = np.expm1(oof), np.expm1(test)
    return np.clip(oof, 0, None), np.clip(test, 0, None)

def main():
    train_ids, test_ids, y, raw_train, raw_test, manifest = load_feature_archive()
    print(f"Archive model: {manifest['model_name']} | train: {len(train_ids)} | test: {len(test_ids)} | views: {len(raw_train)}")
    train_views, test_views = preprocess_views(raw_train, raw_test)
    outer_oof, outer_test = [], []
    for folds in CFG.ENSEMBLE_FOLDS:
        print(f"\nOuter ensemble run: n_folds={folds}")
        pipeline_oof, pipeline_test = [], []
        if CFG.USE_LOG_SPACE:
            oof, pred = run_pipeline(folds, train_views, y, test_views, True)
            pipeline_oof.append(oof); pipeline_test.append(pred)
        if CFG.USE_RAW_SPACE:
            oof, pred = run_pipeline(folds, train_views, y, test_views, False)
            pipeline_oof.append(oof); pipeline_test.append(pred)
        combined_oof, combined_test = np.mean(pipeline_oof, axis=0), np.mean(pipeline_test, axis=0)
        print(f"Reference weighted R2: {weighted_r2_metric(y, combined_oof):.5f}")
        outer_oof.append(combined_oof); outer_test.append(combined_test)
    final_oof, final_test = np.mean(outer_oof, axis=0), np.mean(outer_test, axis=0)
    print(f"Mean training reference weighted R2: {weighted_r2_metric(y, final_oof):.5f}")
    predictions = pd.DataFrame(final_test, columns=CFG.target_cols)
    if CFG.USE_POST_PROCESS: predictions = post_process_biomass(predictions)
    rows = []
    for image_id, values in zip(test_ids, predictions[CFG.target_cols].to_numpy()):
        for name, value in zip(CFG.target_cols, values):
            rows.append({"sample_id": f"{image_id}__{name}", "target": value})
    output_dir = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
    path = output_dir / "submission.csv"
    pd.DataFrame(rows).to_csv(path, index=False)
    print(f"Saved submission: {path}")
    print("Training reference scores may be optimistic because meta-model predictions are fitted on these rows.")

if __name__ == "__main__":
    main()